In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
import pandas as pd
import time

driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()))

driver.get("https://www.myscheme.gov.in/search")

time.sleep(8)

schemes = []

pages_to_scrape = 10   # change this to collect more schemes

for page in range(1, pages_to_scrape + 1):

    print("Scraping page:", page)

    time.sleep(5)

    titles = driver.find_elements(By.XPATH, "//h3")
    descriptions = driver.find_elements(By.XPATH, "//p")

    for i in range(len(titles)):
        name = titles[i].text.strip()

        if name != "":
            desc = ""
            if i < len(descriptions):
                desc = descriptions[i].text.strip()

            schemes.append({
                "scheme_name": name,
                "description": desc
            })

    # click next page
    try:
        next_button = driver.find_element(By.XPATH, "//button[contains(@aria-label,'Next')]")
        next_button.click()
    except:
        print("No more pages")
        break

    time.sleep(5)

driver.quit()

df = pd.DataFrame(schemes)

df.drop_duplicates(inplace=True)

df.to_csv("dataste", index=False)

print("Total schemes collected:", len(df))

In [3]:
import pandas as pd
import random

target_groups = [
    "farmers", "women entrepreneurs", "students", "senior citizens",
    "youth", "small business owners", "rural households",
    "urban poor families", "self help groups", "startup founders"
]

sectors = {
    "Agriculture": "Ministry of Agriculture and Farmers Welfare",
    "Education": "Ministry of Education",
    "Healthcare": "Ministry of Health and Family Welfare",
    "Women Welfare": "Ministry of Women and Child Development",
    "Employment": "Ministry of Labour and Employment",
    "Housing": "Ministry of Housing and Urban Affairs",
    "Energy": "Ministry of Power",
    "Startup": "Ministry of Commerce and Industry",
    "Social Welfare": "Ministry of Social Justice and Empowerment",
    "Skill Development": "Ministry of Skill Development and Entrepreneurship"
}

states = [
    "All India","Andhra Pradesh","Tamil Nadu","Karnataka","Telangana",
    "Kerala","Maharashtra","Gujarat","Rajasthan","Uttar Pradesh",
    "Madhya Pradesh","West Bengal"
]

prefix = [
    "Pradhan Mantri","National","Integrated","Chief Minister",
    "Digital India","Smart India","Rural Development","Urban Development"
]

suffix = [
    "Yojana","Mission","Abhiyan","Samriddhi Scheme","Kalyan Program","Development Initiative"
]

benefits_templates = [
    "financial assistance up to ₹50,000",
    "subsidy covering up to 60% of project cost",
    "low-interest loans through partner banks",
    "free skill training and certification",
    "health insurance coverage up to ₹5 lakh",
    "housing assistance for economically weaker sections"
]

schemes = []

for i in range(50000):

    sector = random.choice(list(sectors.keys()))
    ministry = sectors[sector]

    group = random.choice(target_groups)
    state = random.choice(states)

    name = f"{random.choice(prefix)} {sector} {random.choice(suffix)}"

    min_age = random.randint(18,30)
    max_age = random.randint(40,65)

    income = random.choice([
        "annual family income below ₹2 lakh",
        "annual family income below ₹5 lakh",
        "annual family income below ₹8 lakh"
    ])

    description = f"The {name} is designed to promote {sector.lower()} development by providing targeted support to {group} in {state}. The program focuses on improving access to resources, infrastructure, and financial assistance."

    benefits = random.choice(benefits_templates)

    eligibility = f"Applicants must be {group} aged between {min_age} and {max_age} years with {income}. Applicants must also be permanent residents of {state}."

    application = "Applications can be submitted through the official government portal or at designated government service centers."

    schemes.append({
        "scheme_name": name,
        "description": description,
        "benefits": benefits,
        "eligibility": eligibility,
        "application_process": application,
        "target_beneficiaries": group,
        "ministry": ministry,
        "state": state,
        "sector": sector
    })

df = pd.DataFrame(schemes)

df.drop_duplicates(subset="scheme_name", inplace=True)

df.to_csv("dataset.csv", index=False)

print("Dataset created with", len(df), "schemes")

Dataset created with 480 schemes


In [4]:
import pandas as pd

df = pd.read_csv("dataset.csv")

print("Total schemes:", len(df))

Total schemes: 480


In [5]:
def create_rag_text(row):
    return f"""
Scheme Name: {row['scheme_name']}

Sector: {row['sector']}
Ministry: {row['ministry']}
State: {row['state']}

Description:
{row['description']}

Benefits:
{row['benefits']}

Eligibility:
{row['eligibility']}

Target Beneficiaries:
{row['target_beneficiaries']}

Application Process:
{row['application_process']}
"""

df["rag_text"] = df.apply(create_rag_text, axis=1)

In [6]:
df[["scheme_name","rag_text"]].to_json(
    "schemes_rag_documents.json",
    orient="records",
    indent=2
)

In [7]:
df.to_csv("schemes_rag_documents.csv", index=False)

In [9]:
pip install langchain

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.7/111.7 kB 4.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 160.9/160.9 kB 6.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 502.7/502.7 kB 7.6 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 463.6/463.6 kB 7.0 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 347.2/347.2 kB 6.5 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 174.0/174.0 kB 5.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 604.7/604.7 kB 7.2 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.5/90.5 kB 5.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.5/50.5 kB 4.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 7.4 MB/s eta 0:00:00a 0:00:01m
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 378.3/378.3 kB 8.4 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.5/73.5 kB

In [22]:
documents = []

for text in df["rag_text"]:
    
    chunks = [text[i:i+500] for i in range(0, len(text), 500)]
    
    documents.extend(chunks)

print("Total chunks:", len(documents))

Total chunks: 960


In [23]:
chunk_df = pd.DataFrame({"text": documents})

chunk_df.to_json("rag_chunks.json", orient="records", indent=2)

print("Chunks saved")

Chunks saved


In [24]:
pip install sentence-transformers chromadb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 494.2/494.2 kB 5.9 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.0/20.0 MB 7.5 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.4/78.4 kB 4.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.7/8.7 MB 7.2 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.4/79.4 MB 6.5 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 22.4/22.4 MB 7.1 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 6.5 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 612.9/612.9 kB 7.2 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 310.5/310.5 kB 7.3 MB/s eta 0:00:0000:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 4.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.8/68.8 kB 6.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 

In [25]:
import json

with open("rag_chunks.json", "r") as f:
    data = json.load(f)

texts = [item["text"] for item in data]

print("Total chunks:", len(texts))

Total chunks: 960


In [26]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

embeddings = model.encode(texts)

print("Embeddings created:", len(embeddings))

/Users/hemanth/.pyenv/versions/3.10.10/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9688.79it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embeddings created: 960


In [27]:
import chromadb

client = chromadb.Client()

collection = client.create_collection("government_schemes")

for i, text in enumerate(texts):

    collection.add(
        documents=[text],
        embeddings=[embeddings[i].tolist()],
        ids=[str(i)]
    )

print("Vector database created successfully")

Vector database created successfully


In [28]:
query = "schemes for farmers in agriculture"

query_embedding = model.encode([query])[0]

results = collection.query(
    query_embeddings=[query_embedding.tolist()],
    n_results=3
)

print(results["documents"])

[['\nScheme Name: Rural Development Agriculture Yojana\n\nSector: Agriculture\nMinistry: Ministry of Agriculture and Farmers Welfare\nState: Maharashtra\n\nDescription:\nThe Rural Development Agriculture Yojana is designed to promote agriculture development by providing targeted support to farmers in Maharashtra. The program focuses on improving access to resources, infrastructure, and financial assistance.\n\nBenefits:\nfinancial assistance up to ₹50,000\n\nEligibility:\nApplicants must be farmers aged between ', '\nScheme Name: Urban Development Agriculture Yojana\n\nSector: Agriculture\nMinistry: Ministry of Agriculture and Farmers Welfare\nState: Uttar Pradesh\n\nDescription:\nThe Urban Development Agriculture Yojana is designed to promote agriculture development by providing targeted support to rural households in Uttar Pradesh. The program focuses on improving access to resources, infrastructure, and financial assistance.\n\nBenefits:\nfinancial assistance up to ₹50,000\n\nEligib